Exploring Conversation Chains and Memory with LCEL

In [1]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

In [3]:
from langchain_core.runnables import RunnablePassthrough
runnable = RunnablePassthrough()
runnable.invoke("hello")

'hello'

In [4]:
# Let see realtime uses, lets write a simple chain which takes input at runtime

from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model_name='gpt-4o')

from langchain_core.prompts import ChatPromptTemplate

query='{query}'

prompt=ChatPromptTemplate.from_template(query)

In [5]:
# Approach 1 to invoke llm
llchain=(prompt | llm)
response=llchain.invoke({'query': 'Explain gen ai in 10 words'})
response.content

'AI generating human-like content, enhancing creativity and decision-making processes.'

In [7]:
# Let say the input query is coming from some transformation then we can use Runnable to recive the input query and pass to next step but 
# lanchain chains requres dictonary so {'query':RunnablePassthrough()} will create that dictinary as {'query': 'Explain gen ai in 10 words'}
# Approach 2 to invoke llm
user_query ='Explain gen ai in 10 words'
llchain=({'query':RunnablePassthrough()} |
        prompt | llm)

response=llchain.invoke(user_query)
response.content

'AI that creates content and learns from data patterns autonomously.'

In [8]:
user_query ='Explain gen ai in 10 words'

llchain=(RunnablePassthrough()|
        prompt | llm)

response=llchain.invoke({'query':user_query})
response.content

'Generative AI creates content using machine learning and artificial intelligence.'

In [11]:
from langchain_core.runnables import RunnableLambda

In [9]:
# Lets say we have process the input query and only need to pass the first token in the input to LLM, so we have written python function for that,
# now we can not call this function directly inside LangChain chain for the we need to use RunnableLambda
def clean_query(data: dict):
    text = data["input"]  
    return text.split()[0]

In [15]:
# Note the when called clean_query function we have passed any input to it mean we have not used as clean_query(input_var_name) 
# whatever value will be coming to langchain chain will be passed to "clean_query" function and RunnablePassthrough.assign will assign that value 
# to query variable

llchain = (RunnablePassthrough.assign(query=RunnableLambda(clean_query))
    | prompt
    | llm
)

user_query1 = "Genai and ML"

response = llchain.invoke({'input':user_query1})

print(response.content)

It looks like you're referring to "GenAI," which is short for "Generative AI." Generative AI refers to a class of artificial intelligence models designed to generate new content. This content can range from text, images, music, and more, based on the data they were trained on. Some of the popular applications of generative AI include language models like GPT-3, DALL-E for image generation, and deepfake technology for video synthesis. These tools can be used for a wide array of applications, from content creation and artistic endeavors to more practical uses like data augmentation and simulation. If you have any specific questions or need further details on generative AI, feel free to ask!


In [14]:
# You can not do it this way if you notice chain runs successfuly. chain provides response for query # "Genai and ML" 

llchain = ({"input": RunnablePassthrough()}|
   {'query': RunnablePassthrough(query=RunnableLambda(clean_query))}
    | prompt
    | llm
)

user_query1 = "Genai and ML"

response = llchain.invoke(user_query1)

print(response.content)

"GenAI" typically refers to "Generative AI," a subset of artificial intelligence that focuses on generating content, such as text, images, music, and more, often through the use of sophisticated algorithms and models. Machine Learning (ML), on the other hand, is a broader domain of AI that involves training algorithms on data to make predictions or decisions without being explicitly programmed for specific tasks.

**Key Concepts:**

1. **Generative AI (GenAI):**
   - **Generative Models:** These models can create data samples from learned data distributions. Common generative models include Variational Autoencoders (VAEs), Generative Adversarial Networks (GANs), and Transformer-based models (like GPT).
   - **Applications:** Content creation (text, image, sound), style transfer, data augmentation, and simulations, among others.

2. **Machine Learning (ML):**
   - **Supervised Learning:** Involves training a model on labeled data, where the correct output is provided, such as in classif

In [16]:
# Lets start with ConversationBufferMemory

from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import ConversationBufferMemory




In [17]:
SYS_PROMPT = """Act as a helpful assistant and give brief answers"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{query}"),
    ]
)

memory = ConversationBufferMemory(return_messages=True)

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_10781/159450210.py:10: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages=True)


In [18]:
# function to get historical conversation messages from the memory
memory.load_memory_variables({})

{'history': []}

In [19]:
# lets create a function now which returns the list of messages from memory
def get_memory_messages(query):
    return memory.load_memory_variables(query)['history']

In [20]:

get_memory_messages('What are the first four colors of a rainbow?')

[]

In [21]:
# testing out the function with a runnable lambda which will go into our chain
# this returns the history but we also need to send our current query to the prompt
RunnableLambda(get_memory_messages).invoke({'query': 'What are the first four colors of a rainbow?'})

[]

In [22]:
# we use a runnable passthrough to pass our current query untouched
# along with the history messages to the next step in the chain
RunnablePassthrough.assign(
        history=RunnableLambda(get_memory_messages)
    ).invoke({'query': 'What are the first four colors of a rainbow?'})

{'query': 'What are the first four colors of a rainbow?', 'history': []}

Here RunnableLambda will allow you to run your python function and output of that function will be assignd to "history" variable. but if you look at your prompt varaibale it require to inputs history and query, now RunnablePassthrough will help you to pass the query and history varianle to the next step in the cahin which is prompt. you can see above when we invoked RunnablePassthrough it returned a dictonary of two variables (query and history) which is passsed to next step in chain.

In [23]:
# creating our conversation chain now
def get_memory_messages(query):
    return memory.load_memory_variables(query)['history']

conversation_chain = (
    RunnablePassthrough.assign(
        history=RunnableLambda(get_memory_messages)
    ) # sends current query (input by user at runtime) and history messages to next step
      |
    prompt # creates prompt using the previous two variables
      |
    chatgpt # generates response using the prompt from previous step
)

In [24]:
query = {'query': 'What are the first four colors of a rainbow?'}
response = conversation_chain.invoke(query)
response

AIMessage(content='The first four colors of a rainbow are red, orange, yellow, and green.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 30, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_939a9d4290', 'id': 'chatcmpl-E0AEWnmaJGmmYBgwJ3NNV1PLaDTU3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4d48-c76c-7902-abea-2e98acb366e7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 17, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [25]:
print(response.content)

The first four colors of a rainbow are red, orange, yellow, and green.


In [26]:
# Still memeory is not updated with current query
memory.load_memory_variables({})

{'history': []}

In [27]:
# Lets save the current conversation to memory. memory itself will take care of adding HumanMessage and AIMessage. we just need 
# to apps user query and LLM response to this save_context functon.
memory.save_context(query, {"output": response.content})

In [28]:
memory.load_memory_variables({})

{'history': [HumanMessage(content='What are the first four colors of a rainbow?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The first four colors of a rainbow are red, orange, yellow, and green.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [29]:
query = {'query': 'and the other 3?'}
response = conversation_chain.invoke(query)
memory.save_context(query, {"output": response.content}) # remember to save your current conversation in memory
print(response.content)

The other three colors of a rainbow are blue, indigo, and violet.


In [30]:
query = {'query': 'and the other 3?'}
response = conversation_chain.invoke(query)
memory.save_context(query, {"output": response.content}) # remember to save your current conversation in memory
print(response.content)

There are no additional colors in a standard rainbow; it consists of seven colors in total: red, orange, yellow, green, blue, indigo, and violet.


In [31]:
memory.load_memory_variables({})

{'history': [HumanMessage(content='What are the first four colors of a rainbow?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The first four colors of a rainbow are red, orange, yellow, and green.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='and the other 3?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The other three colors of a rainbow are blue, indigo, and violet.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='and the other 3?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='There are no additional colors in a standard rainbow; it consists of seven colors in total: red, orange, yellow, green, blue, indigo, and violet.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

Conversation Chains with VectorStoreRetrieverMemory

VectorStoreRetrieverMemory stores historical conversation messages in a vector store and queries the top-K most "relevant" history messages every time it is called.

This differs from most of the other Memory classes in that it doesn't explicitly track the order of interactions but retrieves history based on embedding similarity to the current question or prompt.

In this case, the "docs" are previous conversation snippets. This can be useful to refer to relevant pieces of information that the AI was told earlier in the conversation.

In [32]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [33]:
from langchain_chroma import Chroma

# create empty vector DB
chroma_db = Chroma(collection_name='history_db',
                   embedding_function=openai_embed_model)

In [35]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import VectorStoreRetrieverMemory

In [36]:
SYS_PROMPT = """Act as a helpful assistant and give brief answers"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{query}"),
    ]
)

In [37]:

# load 2 most similar conversation messages from vector db history for each new message \ prompt
# this uses cosine embedding similarity to load the top 2 similar messgages to the input prompt \ query
retriever = chroma_db.as_retriever(search_type="similarity",
                                   search_kwargs={"k": 2})
memory = VectorStoreRetrieverMemory(retriever=retriever, return_messages=True)

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_10781/3963828392.py:5: LangChainDeprecationWarning: The class `VectorStoreRetrieverMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = VectorStoreRetrieverMemory(retriever=retriever, return_messages=True)


In [38]:
memory

VectorStoreRetrieverMemory(retriever=VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x1514fc1a0>, search_kwargs={'k': 2}), exclude_input_keys=())

In [39]:
# creating our conversation chain now
def get_memory_messages(query):
    return [memory.load_memory_variables(query)['history']]

In [40]:
conversation_chain = (
    RunnablePassthrough.assign(
        history=RunnableLambda(get_memory_messages)
    ) # sends current query (input by user at runtime) and history messages to next step
      |
    prompt # creates prompt using the previous two variables
      |
    chatgpt # generates response using the prompt from previous step
)

In [41]:
query = {'query': 'Tell me about AI'}
response = conversation_chain.invoke(query)
memory.save_context(query, {"output": response.content}) # remember to save your current conversation in memory
print(response.content)

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines programmed to think and learn. It encompasses various technologies, including machine learning, natural language processing, and robotics. AI can perform tasks such as speech recognition, decision-making, and problem-solving, and is used in applications like virtual assistants, autonomous vehicles, and data analysis. AI continues to evolve, raising both opportunities and ethical considerations.
